# 00b. Reference setup: GENCODE GRCh38 and Salmon index

このノートブックは、FASTQを遺伝子・転写産物に対応づけるための「参照辞書」を作る段階である。

**この段階が答える問い**

- RNA-seq readを何に照合するのか。
- Salmonが使うindexはどこにあるのか。
- transcript単位の結果をgene単位にまとめる対応表はどこにあるのか。

**入力**

- GENCODE human release 49 の transcript FASTA
- 同じreleaseの annotation GTF

**出力**

- `reference/gencode_grch38/gencode.v49.transcripts.fa.gz`
- `reference/gencode_grch38/gencode.v49.annotation.gtf.gz`
- `reference/gencode_grch38/salmon_index/`
- `reference/gencode_grch38/tx2gene.tsv`

**次に進む条件**

- 上の4つの出力が存在する。
- `config/analysis_config.json` の `reference` 欄がこの4つを指している。


## 初心者向けの説明

FASTQには短い配列readしか入っていない。readだけを見ても「どの遺伝子から来たreadか」は分からない。

そこで、既知のヒト転写産物配列を集めたFASTAを使って、Salmon用のindexを作る。これは日本語の文章を辞書で引く準備に近い。

GTFは、transcriptとgeneの対応表を作るために使う。Salmonの最初の結果はtranscript単位なので、DESeq2で扱いやすいgene単位へ集計するために `tx2gene.tsv` が必要である。


In [1]:
from pathlib import Path
import gzip
import json
import re
import shutil
import subprocess

PROJECT_DIR = Path("/Users/yusuke_tateishi/Documents/RNA_seq").resolve()
CONFIG_PATH = PROJECT_DIR / "config" / "analysis_config.json"
REFERENCE_DIR = PROJECT_DIR / "reference" / "gencode_grch38"
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

GENCODE_RELEASE = "49"
TRANSCRIPT_FASTA_NAME = f"gencode.v{GENCODE_RELEASE}.transcripts.fa.gz"
GTF_NAME = f"gencode.v{GENCODE_RELEASE}.annotation.gtf.gz"
GENOME_FASTA_NAME = "GRCh38.primary_assembly.genome.fa.gz"

TRANSCRIPT_FASTA_PATH = REFERENCE_DIR / TRANSCRIPT_FASTA_NAME
GTF_PATH = REFERENCE_DIR / GTF_NAME
GENOME_FASTA_PATH = REFERENCE_DIR / GENOME_FASTA_NAME
TX2GENE_PATH = REFERENCE_DIR / "tx2gene.tsv"
SALMON_INDEX_DIR = REFERENCE_DIR / "salmon_index"
STAR_INDEX_DIR = REFERENCE_DIR / "star_index"

TRANSCRIPT_FASTA_URL = f"https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_{GENCODE_RELEASE}/{TRANSCRIPT_FASTA_NAME}"
GTF_URL = f"https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_{GENCODE_RELEASE}/{GTF_NAME}"
GENOME_FASTA_URL = f"https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_{GENCODE_RELEASE}/{GENOME_FASTA_NAME}"

print("Reference directory:", REFERENCE_DIR)
print("Transcript FASTA URL:", TRANSCRIPT_FASTA_URL)
print("GTF URL:", GTF_URL)
print("Genome FASTA URL:", GENOME_FASTA_URL)


Reference directory: /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38
Transcript FASTA URL: https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.transcripts.fa.gz
GTF URL: https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.annotation.gtf.gz
Genome FASTA URL: https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/GRCh38.primary_assembly.genome.fa.gz


## Step 1: 参照ファイルをダウンロードする

このセルはインターネットから大きめのファイルを取得する。初回だけ `RUN_DOWNLOAD = True` にしよう。

途中で止まっても、`curl -C -` を使うので再開できる。


In [2]:
RUN_DOWNLOAD = False

def download_with_curl(url: str, out_path: Path) -> None:
    curl = shutil.which("curl")
    if curl is None:
        raise RuntimeError("curl was not found.")
    command = [curl, "-L", "--fail", "-C", "-", "-o", str(out_path), url]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

if RUN_DOWNLOAD:
    download_with_curl(TRANSCRIPT_FASTA_URL, TRANSCRIPT_FASTA_PATH)
    download_with_curl(GTF_URL, GTF_PATH)
    download_with_curl(GENOME_FASTA_URL, GENOME_FASTA_PATH)
else:
    print("Set RUN_DOWNLOAD = True to download the GENCODE FASTA, GTF, and Genome FASTA files.")


Set RUN_DOWNLOAD = True to download the GENCODE FASTA, GTF, and Genome FASTA files.


## Step 2: ダウンロードできたか確認する

ここではファイルサイズとgzip形式を確認する。中身の生物学的評価ではなく、「壊れていないか」の最低限の確認である。


In [3]:
for path in [TRANSCRIPT_FASTA_PATH, GTF_PATH, GENOME_FASTA_PATH]:
    if path.exists():
        print(path.relative_to(PROJECT_DIR), f"{path.stat().st_size / 1024 / 1024:.1f} MB")
        with gzip.open(path, "rt") as handle:
            first_line = handle.readline().strip()
        print("  first line:", first_line[:120])
    else:
        print("missing:", path.relative_to(PROJECT_DIR))


reference/gencode_grch38/gencode.v49.transcripts.fa.gz 134.9 MB
  first line: >ENST00000832824.1|ENSG00000290825.2|-|-|DDX11L16-260|DDX11L16|1379|lncRNA|
reference/gencode_grch38/gencode.v49.annotation.gtf.gz 89.0 MB
  first line: ##description: evidence-based annotation of the human genome (GRCh38), version 49 (Ensembl 115)
missing: reference/gencode_grch38/GRCh38.primary_assembly.genome.fa.gz


## Step 3: Salmon indexを作る

Salmon indexは、SalmonがFASTQ readを高速に照合するための専用データベースである。

この処理は少し時間がかかる。初回だけ `RUN_BUILD_SALMON_INDEX = True` にしよう。


### `salmon index` コマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `-t FILE` | Transcript FASTA（`gencode.v*.transcripts.fa.gz`）を指定する。index構築の元データである。 |
| `-i DIR` | 作成したindexの出力先ディレクトリ（`salmon_index/`）を指定する。 |
| `--gencode` | GENCODEのFASTAヘッダー形式（`|` 区切り）を正しく解析するフラグである。GENCODEを使う場合は必須。 |
| `-p N` | 並列スレッド数。index構築に使うCPU数である。 |

Salmon indexは、FASTQ readとTranscriptの対応を高速に照合するための内部データ構造（k-merハッシュ）である。一度作れば同じreferenceへの全サンプル定量で使い回せる。


In [4]:
RUN_BUILD_SALMON_INDEX = False
THREADS = 8

if RUN_BUILD_SALMON_INDEX:
    salmon = shutil.which("salmon")
    if salmon is None:
        raise RuntimeError("salmon was not found. Activate the rna-seq environment first.")
    if not TRANSCRIPT_FASTA_PATH.exists():
        raise FileNotFoundError(TRANSCRIPT_FASTA_PATH)
    command = [
        salmon,
        "index",
        "-t",
        str(TRANSCRIPT_FASTA_PATH),
        "-i",
        str(SALMON_INDEX_DIR),
        "--gencode",
        "-p",
        str(THREADS),
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
else:
    print("Set RUN_BUILD_SALMON_INDEX = True after the transcript FASTA is downloaded.")


Set RUN_BUILD_SALMON_INDEX = True after the transcript FASTA is downloaded.


## Step 3b: STAR indexを作る

STAR indexは、STARがFASTQ readをGenome領域にmappingするためのGenomeインデックスデータベースである。

> [!WARNING]
> **メモリ消費量について**
> ヒトGenome（GRCh38）に対するSTARのインデックス構築には、**通常30GB以上のRAM（メモリ）**が必要である。
> もし個人のPCなどでメモリが足りない場合は、このステップをスキップし、十分なリソースがある共有サーバー等でインデックスを構築しよう。

この処理には少し時間がかかる（約30分〜1時間）。初回だけ `RUN_BUILD_STAR_INDEX = True` にしよう。


### STAR indexコマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `--runMode genomeGenerate` | index構築モード。通常のmapping時は `alignReads`（デフォルト）を使う。 |
| `--genomeDir DIR` | index出力ディレクトリ（`star_index/`）を指定する。 |
| `--genomeFastaFiles FILE` | Genome FASTA（`GRCh38.primary_assembly.genome.fa`）を指定する。 |
| `--sjdbGTFfile FILE` | splice junction（スプライス部位）情報を持つGTFファイルを指定する。 |
| `--sjdbOverhang N` | `read長 - 1` を指定する（例: 150 bpのread → `149`）。スプライスジャンクションindex構築の精度に影響する。 |
| `--runThreadN N` | 並列スレッド数。 |

STARは**Genome-based mapping**（Genome全体へのsplice-aware alignment）を行う。Salmonと異なり、BAMファイルを出力するため、featureCountsによるgeneごとのcount集計が可能である。
ヒトGenome（GRCh38）のindex構築には通常**30 GB以上のRAM**が必要である。


In [5]:
RUN_BUILD_STAR_INDEX = False
THREADS = 8
SJDB_OVERHANG = 149

if RUN_BUILD_STAR_INDEX:
    star = shutil.which("STAR")
    if star is None:
        raise RuntimeError("STAR was not found. Activate the rna-seq environment first.")
    if not GENOME_FASTA_PATH.exists():
        raise FileNotFoundError(GENOME_FASTA_PATH)
    if not GTF_PATH.exists():
        raise FileNotFoundError(GTF_PATH)

    STAR_INDEX_DIR.mkdir(parents=True, exist_ok=True)

    # STAR genomeGenerate requires decompressed fasta and gtf
    decompressed_fasta = GENOME_FASTA_PATH.with_suffix("")
    decompressed_gtf = GTF_PATH.with_suffix("")

    print(f"Decompressing {GENOME_FASTA_PATH.name}...")
    with gzip.open(GENOME_FASTA_PATH, "rb") as f_in, decompressed_fasta.open("wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

    print(f"Decompressing {GTF_PATH.name}...")
    with gzip.open(GTF_PATH, "rb") as f_in, decompressed_gtf.open("wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

    try:
        command = [
            star,
            "--runMode", "genomeGenerate",
            "--genomeDir", str(STAR_INDEX_DIR),
            "--genomeFastaFiles", str(decompressed_fasta),
            "--sjdbGTFfile", str(decompressed_gtf),
            "--sjdbOverhang", str(SJDB_OVERHANG),
            "--runThreadN", str(THREADS),
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, check=True)
    finally:
        # Clean up decompressed files to save disk space
        if decompressed_fasta.exists():
            print(f"Removing temporary decompressed FASTA: {decompressed_fasta}")
            decompressed_fasta.unlink()
        if decompressed_gtf.exists():
            print(f"Removing temporary decompressed GTF: {decompressed_gtf}")
            decompressed_gtf.unlink()
else:
    print("Set RUN_BUILD_STAR_INDEX = True after the genome FASTA and GTF are downloaded.")


Set RUN_BUILD_STAR_INDEX = True after the genome FASTA and GTF are downloaded.


## Step 4: `tx2gene.tsv`を作る

`tx2gene.tsv` は2列以上の表である。

- `Name`: Salmonの `quant.sf` に出るtranscript ID
- `gene_id`: そのtranscriptをgene単位にまとめるためのID

ここでは後のclusterProfilerを分かりやすくするため、`gene_id` 列には基本的に遺伝子シンボルを入れる。もし将来Ensembl IDで解析したい場合は、`gene_id_type` を `ENSEMBL` に変える必要がある。


In [6]:
RUN_BUILD_TX2GENE = False

if RUN_BUILD_TX2GENE:
    if not GTF_PATH.exists():
        raise FileNotFoundError(GTF_PATH)

    seen_transcripts = set()
    written = 0
    with gzip.open(GTF_PATH, "rt") as gtf, TX2GENE_PATH.open("w", encoding="utf-8") as out:
        out.write("Name\tgene_id\tensembl_gene_id\n")
        for line in gtf:
            if line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) != 9 or fields[2] != "transcript":
                continue
            attributes = fields[8]
            attrs = dict(re.findall(r'(\S+) "([^"]+)"', attributes))
            transcript_id = attrs.get("transcript_id")
            ensembl_gene_id = attrs.get("gene_id")
            gene_name = attrs.get("gene_name") or ensembl_gene_id
            if not transcript_id or not gene_name:
                continue
            if transcript_id in seen_transcripts:
                continue
            seen_transcripts.add(transcript_id)
            out.write(f"{transcript_id}\t{gene_name}\t{ensembl_gene_id}\n")
            written += 1

    print("Wrote:", TX2GENE_PATH.relative_to(PROJECT_DIR))
    print("Transcript rows:", written)
else:
    print("Set RUN_BUILD_TX2GENE = True after the GTF file is downloaded.")


Set RUN_BUILD_TX2GENE = True after the GTF file is downloaded.


In [7]:
if TX2GENE_PATH.exists():
    import pandas as pd

    tx2gene = pd.read_csv(TX2GENE_PATH, sep="\t")
    print(tx2gene.shape)
    display(tx2gene.head())
else:
    print("tx2gene.tsv is not created yet.")


(507365, 4)


,transcript_id,transcript_name,gene_name,gene_id
0,ENST00000832824.1,DDX11L16-260,DDX11L16,ENSG00000290825.2
1,ENST00000832825.1,DDX11L16-261,DDX11L16,ENSG00000290825.2
2,ENST00000832826.1,DDX11L16-262,DDX11L16,ENSG00000290825.2
3,ENST00000832827.1,DDX11L16-263,DDX11L16,ENSG00000290825.2
4,ENST00000832828.1,DDX11L16-264,DDX11L16,ENSG00000290825.2


## Step 5: configに参照パスを書き込む

`analysis_config.json` の `reference` 欄に、今作った参照ファイルの場所を書きる。相対パスで書くので、別のノートブックからも同じように読める。


In [8]:
RUN_UPDATE_CONFIG = False

if RUN_UPDATE_CONFIG:
    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    config["organism"] = "human"
    config["gene_id_type"] = "SYMBOL"
    config["reference"] = {
        "gencode_release": GENCODE_RELEASE,
        "transcript_fasta": str(TRANSCRIPT_FASTA_PATH.relative_to(PROJECT_DIR)),
        "salmon_index": str(SALMON_INDEX_DIR.relative_to(PROJECT_DIR)),
        "star_index": str(STAR_INDEX_DIR.relative_to(PROJECT_DIR)),
        "tx2gene_path": str(TX2GENE_PATH.relative_to(PROJECT_DIR)),
        "gtf_path": str(GTF_PATH.relative_to(PROJECT_DIR)),
        "genome_fasta": str(GENOME_FASTA_PATH.relative_to(PROJECT_DIR)),
    }
    CONFIG_PATH.write_text(json.dumps(config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print("Updated:", CONFIG_PATH.relative_to(PROJECT_DIR))
else:
    print("Set RUN_UPDATE_CONFIG = True after the reference files and index are ready.")


Set RUN_UPDATE_CONFIG = True after the reference files and index are ready.


## 最終確認

ここが全部 `True` になったら、各mappingルートに応じた定量処理へ進める。


In [9]:
readiness = {
    "transcript_fasta": TRANSCRIPT_FASTA_PATH.exists(),
    "genome_fasta": GENOME_FASTA_PATH.exists(),
    "gtf": GTF_PATH.exists(),
    "tx2gene": TX2GENE_PATH.exists(),
    "salmon_index_dir": SALMON_INDEX_DIR.exists(),
    "star_index_dir": STAR_INDEX_DIR.exists(),
    "config": CONFIG_PATH.exists(),
}
readiness


{'transcript_fasta': True,
 'genome_fasta': False,
 'gtf': True,
 'tx2gene': True,
 'salmon_index_dir': True,
 'star_index_dir': True,
 'config': True}

**よくある間違い**

- FASTAとGTFのreleaseを混ぜる。例: FASTAはv49、GTFはv48。
- Salmon indexやSTAR indexを作らずに発現定量へ進む。
- `tx2gene.tsv` を作らずにgene count matrixを作ろうとする。

**小さい練習**

`readiness` のどれが `False` かを見て、次に実行すべき `RUN_*` フラグを1つだけ選ぼう。
